In [1]:
import os, random, argparse, itertools, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from _utils import MMDataset, clustering_acc, overall_performance_report

class DUCMME(nn.Module):
    def __init__(self, embed_dim=200, num_samples=10000, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], n_clusters=10, alpha=1.0):
        super(DUCMME, self).__init__()
        self.embed_dim = embed_dim; self.num_samples = num_samples; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.n_clusters = n_clusters; self.alpha = alpha
        # 1. Multi-view Feature Extraction by Fusion-Net
        self.fusion_net_encoder = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                               nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)]) # encode each view
        self.fusion_net_mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=10, batch_first=True) # batch_first=True: (batch_size, seq_len, hidden_dim)
        self.fusion_net_linear = nn.Linear(3*embed_dim, embed_dim) # linear projection of the fused encoded features
        # 2. Uncertainty-Aware Reconstruction by Reconstruction-Net and Uncertainty-Net
        self.reconstruct_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # reconstruct each view
        self.uncertainty_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # predict uncertainty for each view
        # 3. Deep Embedding Clustering by DEC
        self._cluster_centers = nn.Parameter(torch.Tensor(self.n_clusters, self.embed_dim))
        nn.init.xavier_uniform_(self._cluster_centers.data)
        
    def forward_embedding(self, x):
        encoded_output_list = [self.fusion_net_encoder[i](x[i]) for i in range(self.num_views)] # encode each view
        encoded_output_list = torch.stack(encoded_output_list, dim=1) # stack the encoded features from all views, (batch_size, num_views, embed_dim)
        encoded_output_list, _ = self.fusion_net_mha(encoded_output_list, encoded_output_list, encoded_output_list) # fuse the encoded features from all views by a multihead attention, (batch_size, num_views, embed_dim)
        encoded_output_list = encoded_output_list.contiguous().view(encoded_output_list.shape[0], -1) # flatten the encoded features, (batch_size, num_views*embed_dim)
        embedding = self.fusion_net_linear(encoded_output_list) # linear projection of the fused encoded features
        return embedding # get the embedding of the latent space H, (batch_size, embed_dim)

    def forward_uncertainty_aware_reconstruction(self, x):
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        reconstructions = [self.reconstruct_net_list[i](embedding) for i in range(self.num_views)] # reconstruct each view
        uncertainties = [self.uncertainty_net_list[i](embedding) for i in range(self.num_views)] # predict uncertainty for each view
        return reconstructions, uncertainties
        
    def forward_similarity_matrix_q(self, x): # calculate the similarity matrix q using t-distribution
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        q = 1.0 / (1.0 + torch.sum((embedding.unsqueeze(1) - self._cluster_centers) ** 2, dim=2) / self.alpha) # shape: [batch_size, n_clusters]
        q = q ** ((self.alpha + 1.0) / 2.0) # , shape: [batch_size, n_clusters]
        q = q / torch.sum(q, dim=1, keepdim=True) # Normalize q to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return q, embedding # q can be regarded as the probability of the sample belonging to each cluster
    
    @property
    def cluster_centers(self):
        return self._cluster_centers.data.detach().cpu().numpy() # shape: (n_clusters, embed_dim)
    
    @cluster_centers.setter
    def cluster_centers(self, centers): # shape: (n_clusters, embed_dim)
        centers = torch.tensor(centers, dtype=torch.float32, device=self._cluster_centers.device)
        self._cluster_centers.data.copy_(centers) # copy the cluster centers to the model, set the cluster centers to the new cluster centers
        
    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0) # shape: [batch_size, n_clusters]
        p = weight / torch.sum(weight, dim=1, keepdim=True) # Normalize p to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return p.clone().detach()
    
    def reconstruction_loss(self, x):
        x_rec, _ = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        return sum([F.mse_loss(x_rec[v], x[v], reduction='mean') for v in range(self.num_views)]) # sum the losses from all views
    
    def uncertainty_aware_reconstruction_loss(self, x):
        x_rec, log_sigma_2 = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        # Clip log_sigma_2 to prevent numerical instability (exp(-log_sigma_2) overflow/underflow)
        # Clamp to reasonable range: -10 to 10, which corresponds to sigma^2 from exp(-10) to exp(10)
        # log_sigma_2 = [torch.clamp(log_sigma_2[v], min=-10.0, max=10.0) for v in range(self.num_views)] # shape: [num_views, batch_size, feature_dim] for numerically stable computation
        return sum([0.5 * torch.mean((x_rec[v] - x[v])**2 * torch.exp(-log_sigma_2[v]) + log_sigma_2[v]) for v in range(self.num_views)]) # uncertainty is equal to log_sigma_2
    
    def clustering_loss(self, x, p):
        q, _ = self.forward_similarity_matrix_q(x) # shape: [batch_size, n_clusters]
        return F.kl_div(q.log(), p, reduction='batchmean') # shape: ()

In [2]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_sc_multiomics/TEA/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict = dataset.get_label_to_name()

dataloader = torch.utils.data.DataLoader(dataset, batch_size=data_samples, shuffle=True)
model = DUCMME(embed_dim=20, feature_dims=data_features, num_views=data_views, hidden_dims=[512, 512, 512], num_samples=data_samples, n_clusters=data_categories, alpha=1.0).to(device)
## === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
print("\n=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===")
print("Basic reconstruction training...")
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
losses_convergence_plot_reconstruction = []
for epoch in range(200):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Pretraining completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
print("Uncertainty-aware reconstruction finetuning...")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_convergence_plot_reconstruction_finetuning = []
for epoch in range(100):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.uncertainty_aware_reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction_finetuning.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Finetuning completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")

## === Stage 2: Deep Embedding Clustering by DEC ===
print("\n=== Stage 2: Deep Embedding Clustering ===")
print("Cluster center initialization...")
model.eval()
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Cluster center initialization completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
model.cluster_centers = kmeans.cluster_centers_ # shape: (n_clusters, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
losses = []
acc_convergence_plot = []
nmi_convergence_plot = []
asw_convergence_plot = []
ari_convergence_plot = []
embeddings_dict = {}
preds_dict = {}
for epoch in range(150):
    # Update target distribution periodically
    if epoch % 1 == 0:
        model.eval()
        with torch.no_grad():
            q, embedding = model.forward_similarity_matrix_q(data)
            p = model.target_distribution(q)
        y_pred = torch.argmax(q, dim=1).cpu().numpy()
        acc_val = clustering_acc(label, y_pred)
        nmi_val = normalized_mutual_info_score(label, y_pred)
        asw_val = silhouette_score(embedding.cpu().numpy(), y_pred)
        ari_val = adjusted_rand_score(label, y_pred)
        acc_convergence_plot.append(acc_val)
        nmi_convergence_plot.append(nmi_val)
        asw_convergence_plot.append(asw_val)
        ari_convergence_plot.append(ari_val)
        embeddings_dict[epoch] = embedding
        preds_dict[epoch] = y_pred
        if epoch == 0:
            delta_label = 1.0
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
        else:
            delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
            if delta_label < 0.001:
                convergence_epoch = epoch
                print('Converged, stopping training...')
                # break
    # Training based on the target distribution that is updated periodically
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.clustering_loss(x, p[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
print(f'Final ACC: {clustering_acc(label, y_pred):.4f}')

modality_rna shape: (25517, 50)
modality_protein shape: (25517, 47)
modality_atac shape: (25517, 30)

=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
Basic reconstruction training...
Epoch 0 completed. Average Loss: 1.0746
Epoch 1 completed. Average Loss: 1.4307
Epoch 2 completed. Average Loss: 0.6926
Epoch 3 completed. Average Loss: 0.4292
Epoch 4 completed. Average Loss: 0.5287
Epoch 5 completed. Average Loss: 0.4477
Epoch 6 completed. Average Loss: 0.2914
Epoch 7 completed. Average Loss: 0.2407
Epoch 8 completed. Average Loss: 0.2511
Epoch 9 completed. Average Loss: 0.2271
Epoch 10 completed. Average Loss: 0.1898
Epoch 11 completed. Average Loss: 0.1602
Epoch 12 completed. Average Loss: 0.1315
Epoch 13 completed. Average Loss: 0.1137
Epoch 14 completed. Average Loss: 0.1118
Epoch 15 completed. Average Loss: 0.1128
Epoch 16 completed. Average Loss: 0.1011
Epoch 17 completed. Average Loss: 0.0795
Epoch 18 completed. Average Loss: 0.0660
Epoch 19 completed. Average Loss: 

In [ ]:
# modality_rna shape: (25517, 50)
# modality_protein shape: (25517, 47)
# modality_atac shape: (25517, 30)
# === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
# Basic reconstruction training...
# Pretraining completed. ACC: 0.6273, NMI: 0.6338, ASW: 0.3813, ARI: 0.4993
# Uncertainty-aware reconstruction finetuning...
# Finetuning completed. ACC: 0.6589, NMI: 0.6557, ASW: 0.3290, ARI: 0.5161
# === Stage 2: Deep Embedding Clustering ===
# Cluster center initialization...
# Cluster center initialization completed. ACC: 0.6571, NMI: 0.6547, ASW: 0.3273, ARI: 0.5153
# [Epoch 0] ACC: 0.6571, NMI: 0.6547, ASW: 0.3273, ARI: 0.5153, Delta: 1.0000
# Epoch 0 completed. Average Loss: 0.1801
# [Epoch 1] ACC: 0.7174, NMI: 0.6655, ASW: 0.3217, ARI: 0.5468, Delta: 0.2052
# Epoch 1 completed. Average Loss: 0.1463
# [Epoch 2] ACC: 0.7388, NMI: 0.6722, ASW: 0.3135, ARI: 0.5613, Delta: 0.1012
# Epoch 2 completed. Average Loss: 0.1293
# [Epoch 3] ACC: 0.7288, NMI: 0.6736, ASW: 0.3185, ARI: 0.5642, Delta: 0.0562
# Epoch 3 completed. Average Loss: 0.1195
# [Epoch 4] ACC: 0.7401, NMI: 0.6737, ASW: 0.3270, ARI: 0.5738, Delta: 0.0758
# Epoch 4 completed. Average Loss: 0.1093
# [Epoch 5] ACC: 0.7593, NMI: 0.6784, ASW: 0.3358, ARI: 0.5907, Delta: 0.0749
# Epoch 5 completed. Average Loss: 0.1011
# [Epoch 6] ACC: 0.7582, NMI: 0.6834, ASW: 0.3462, ARI: 0.5989, Delta: 0.0589
# Epoch 6 completed. Average Loss: 0.0975
# [Epoch 7] ACC: 0.7462, NMI: 0.6853, ASW: 0.3609, ARI: 0.5941, Delta: 0.0381
# Epoch 7 completed. Average Loss: 0.0959
# [Epoch 8] ACC: 0.7310, NMI: 0.6859, ASW: 0.3706, ARI: 0.5853, Delta: 0.0271
# Epoch 8 completed. Average Loss: 0.0946
# [Epoch 9] ACC: 0.7189, NMI: 0.6859, ASW: 0.3784, ARI: 0.5776, Delta: 0.0204
# Epoch 9 completed. Average Loss: 0.0932
# [Epoch 10] ACC: 0.7189, NMI: 0.6873, ASW: 0.3849, ARI: 0.5724, Delta: 0.0172
# Epoch 10 completed. Average Loss: 0.0924
# [Epoch 11] ACC: 0.7188, NMI: 0.6895, ASW: 0.3938, ARI: 0.5690, Delta: 0.0145
# Epoch 11 completed. Average Loss: 0.0928
# [Epoch 12] ACC: 0.7180, NMI: 0.6914, ASW: 0.4016, ARI: 0.5662, Delta: 0.0119
# Epoch 12 completed. Average Loss: 0.0944
# [Epoch 13] ACC: 0.7154, NMI: 0.6919, ASW: 0.4090, ARI: 0.5624, Delta: 0.0107
# Epoch 13 completed. Average Loss: 0.0964
# [Epoch 14] ACC: 0.7103, NMI: 0.6920, ASW: 0.4199, ARI: 0.5574, Delta: 0.0134
# Epoch 14 completed. Average Loss: 0.0977
# [Epoch 15] ACC: 0.7041, NMI: 0.6919, ASW: 0.4306, ARI: 0.5534, Delta: 0.0126
# Epoch 15 completed. Average Loss: 0.0979
# [Epoch 16] ACC: 0.6971, NMI: 0.6917, ASW: 0.4426, ARI: 0.5502, Delta: 0.0118
# Epoch 16 completed. Average Loss: 0.0976
# [Epoch 17] ACC: 0.6887, NMI: 0.6913, ASW: 0.4506, ARI: 0.5479, Delta: 0.0112
# Epoch 17 completed. Average Loss: 0.0979
# [Epoch 18] ACC: 0.6828, NMI: 0.6913, ASW: 0.4950, ARI: 0.5471, Delta: 0.0079
# Epoch 18 completed. Average Loss: 0.0994
# [Epoch 19] ACC: 0.6800, NMI: 0.6922, ASW: 0.4987, ARI: 0.5474, Delta: 0.0049
# Epoch 19 completed. Average Loss: 0.1018
# [Epoch 20] ACC: 0.6799, NMI: 0.6928, ASW: 0.4991, ARI: 0.5477, Delta: 0.0032
# Epoch 20 completed. Average Loss: 0.1044
# [Epoch 21] ACC: 0.6835, NMI: 0.6931, ASW: 0.4974, ARI: 0.5483, Delta: 0.0069
# Epoch 21 completed. Average Loss: 0.1070
# [Epoch 22] ACC: 0.6894, NMI: 0.6931, ASW: 0.4499, ARI: 0.5491, Delta: 0.0116
# Epoch 22 completed. Average Loss: 0.1093
# [Epoch 23] ACC: 0.6975, NMI: 0.6937, ASW: 0.4466, ARI: 0.5523, Delta: 0.0145
# Epoch 23 completed. Average Loss: 0.1113
# [Epoch 24] ACC: 0.7055, NMI: 0.6949, ASW: 0.4447, ARI: 0.5571, Delta: 0.0160
# Epoch 24 completed. Average Loss: 0.1124
# [Epoch 25] ACC: 0.7125, NMI: 0.6954, ASW: 0.4445, ARI: 0.5626, Delta: 0.0151
# Epoch 25 completed. Average Loss: 0.1123
# [Epoch 26] ACC: 0.7191, NMI: 0.6966, ASW: 0.4470, ARI: 0.5694, Delta: 0.0141
# Epoch 26 completed. Average Loss: 0.1108
# [Epoch 27] ACC: 0.7218, NMI: 0.6965, ASW: 0.4448, ARI: 0.5734, Delta: 0.0103
# Epoch 27 completed. Average Loss: 0.1079
# [Epoch 28] ACC: 0.7239, NMI: 0.6963, ASW: 0.5025, ARI: 0.5759, Delta: 0.0071
# Epoch 28 completed. Average Loss: 0.1042
# [Epoch 29] ACC: 0.7246, NMI: 0.6956, ASW: 0.5101, ARI: 0.5768, Delta: 0.0058
# Epoch 29 completed. Average Loss: 0.1003
# [Epoch 30] ACC: 0.7241, NMI: 0.6947, ASW: 0.5158, ARI: 0.5765, Delta: 0.0054
# Epoch 30 completed. Average Loss: 0.0967
# [Epoch 31] ACC: 0.7232, NMI: 0.6935, ASW: 0.5206, ARI: 0.5752, Delta: 0.0065
# Epoch 31 completed. Average Loss: 0.0940
# [Epoch 32] ACC: 0.7214, NMI: 0.6926, ASW: 0.4635, ARI: 0.5736, Delta: 0.0069
# Epoch 32 completed. Average Loss: 0.0921
# [Epoch 33] ACC: 0.7193, NMI: 0.6918, ASW: 0.4670, ARI: 0.5716, Delta: 0.0074
# Epoch 33 completed. Average Loss: 0.0905
# [Epoch 34] ACC: 0.7165, NMI: 0.6902, ASW: 0.4685, ARI: 0.5685, Delta: 0.0096
# Epoch 34 completed. Average Loss: 0.0902
# [Epoch 35] ACC: 0.7129, NMI: 0.6882, ASW: 0.4634, ARI: 0.5649, Delta: 0.0097
# Epoch 35 completed. Average Loss: 0.0908
# [Epoch 36] ACC: 0.7071, NMI: 0.6847, ASW: 0.4579, ARI: 0.5593, Delta: 0.0143
# Epoch 36 completed. Average Loss: 0.0928
# [Epoch 37] ACC: 0.6982, NMI: 0.6804, ASW: 0.4571, ARI: 0.5509, Delta: 0.0196
# Epoch 37 completed. Average Loss: 0.0960
# [Epoch 38] ACC: 0.6873, NMI: 0.6764, ASW: 0.4602, ARI: 0.5418, Delta: 0.0240
# Epoch 38 completed. Average Loss: 0.1000
# [Epoch 39] ACC: 0.6718, NMI: 0.6716, ASW: 0.4696, ARI: 0.5303, Delta: 0.0319
# Epoch 39 completed. Average Loss: 0.1047
# [Epoch 40] ACC: 0.6508, NMI: 0.6672, ASW: 0.4841, ARI: 0.5194, Delta: 0.0489
# Epoch 40 completed. Average Loss: 0.1091
# [Epoch 41] ACC: 0.6268, NMI: 0.6635, ASW: 0.4947, ARI: 0.5113, Delta: 0.0594
# Epoch 41 completed. Average Loss: 0.1117
# [Epoch 42] ACC: 0.6277, NMI: 0.6615, ASW: 0.5016, ARI: 0.5086, Delta: 0.0755
# Epoch 42 completed. Average Loss: 0.1132
# [Epoch 43] ACC: 0.6363, NMI: 0.6628, ASW: 0.5026, ARI: 0.5100, Delta: 0.0844
# Epoch 43 completed. Average Loss: 0.1136
# [Epoch 44] ACC: 0.6449, NMI: 0.6669, ASW: 0.5039, ARI: 0.5149, Delta: 0.0833
# Epoch 44 completed. Average Loss: 0.1128
# [Epoch 45] ACC: 0.6563, NMI: 0.6736, ASW: 0.5041, ARI: 0.5230, Delta: 0.0757
# Epoch 45 completed. Average Loss: 0.1110
# [Epoch 46] ACC: 0.6889, NMI: 0.6798, ASW: 0.5001, ARI: 0.5371, Delta: 0.0614
# Epoch 46 completed. Average Loss: 0.1092
# [Epoch 47] ACC: 0.6962, NMI: 0.6832, ASW: 0.5023, ARI: 0.5403, Delta: 0.0464
# Epoch 47 completed. Average Loss: 0.1072
# [Epoch 48] ACC: 0.6852, NMI: 0.6853, ASW: 0.5032, ARI: 0.5341, Delta: 0.0402
# Epoch 48 completed. Average Loss: 0.1067
# [Epoch 49] ACC: 0.6663, NMI: 0.6859, ASW: 0.4992, ARI: 0.5306, Delta: 0.0379
# Epoch 49 completed. Average Loss: 0.1073
# [Epoch 50] ACC: 0.6871, NMI: 0.6885, ASW: 0.4960, ARI: 0.5363, Delta: 0.0320
# Epoch 50 completed. Average Loss: 0.1081
# [Epoch 51] ACC: 0.7022, NMI: 0.6918, ASW: 0.4888, ARI: 0.5442, Delta: 0.0205
# Epoch 51 completed. Average Loss: 0.1088
# [Epoch 52] ACC: 0.7079, NMI: 0.6932, ASW: 0.4894, ARI: 0.5475, Delta: 0.0093
# Epoch 52 completed. Average Loss: 0.1096
# [Epoch 53] ACC: 0.7070, NMI: 0.6931, ASW: 0.4944, ARI: 0.5456, Delta: 0.0082
# Epoch 53 completed. Average Loss: 0.1109
# [Epoch 54] ACC: 0.7001, NMI: 0.6915, ASW: 0.5039, ARI: 0.5400, Delta: 0.0163
# Epoch 54 completed. Average Loss: 0.1125
# [Epoch 55] ACC: 0.6886, NMI: 0.6899, ASW: 0.5085, ARI: 0.5352, Delta: 0.0263
# Epoch 55 completed. Average Loss: 0.1138
# [Epoch 56] ACC: 0.6965, NMI: 0.6898, ASW: 0.5162, ARI: 0.5367, Delta: 0.0340
# Epoch 56 completed. Average Loss: 0.1153
# [Epoch 57] ACC: 0.7149, NMI: 0.6893, ASW: 0.5203, ARI: 0.5420, Delta: 0.0390
# Epoch 57 completed. Average Loss: 0.1175
# [Epoch 58] ACC: 0.7212, NMI: 0.6873, ASW: 0.5179, ARI: 0.5443, Delta: 0.0494
# Epoch 58 completed. Average Loss: 0.1219
# [Epoch 59] ACC: 0.7077, NMI: 0.6827, ASW: 0.5174, ARI: 0.5371, Delta: 0.0647
# Epoch 59 completed. Average Loss: 0.1280
# [Epoch 60] ACC: 0.6732, NMI: 0.6743, ASW: 0.5136, ARI: 0.5217, Delta: 0.0938
# Epoch 60 completed. Average Loss: 0.1351
# [Epoch 61] ACC: 0.6541, NMI: 0.6659, ASW: 0.5074, ARI: 0.5040, Delta: 0.1177
# Epoch 61 completed. Average Loss: 0.1402
# [Epoch 62] ACC: 0.6193, NMI: 0.6643, ASW: 0.5074, ARI: 0.4976, Delta: 0.1428
# Epoch 62 completed. Average Loss: 0.1409
# [Epoch 63] ACC: 0.6770, NMI: 0.6744, ASW: 0.5029, ARI: 0.5231, Delta: 0.1392
# Epoch 63 completed. Average Loss: 0.1332
# [Epoch 64] ACC: 0.7336, NMI: 0.6908, ASW: 0.5061, ARI: 0.5655, Delta: 0.0976
# Epoch 64 completed. Average Loss: 0.1236
# [Epoch 65] ACC: 0.7669, NMI: 0.7064, ASW: 0.5176, ARI: 0.5983, Delta: 0.0573
# Epoch 65 completed. Average Loss: 0.1195
# [Epoch 66] ACC: 0.7833, NMI: 0.7174, ASW: 0.5349, ARI: 0.6164, Delta: 0.0327
# Epoch 66 completed. Average Loss: 0.1189
# [Epoch 67] ACC: 0.7914, NMI: 0.7237, ASW: 0.5486, ARI: 0.6254, Delta: 0.0190
# Epoch 67 completed. Average Loss: 0.1199
# [Epoch 68] ACC: 0.7961, NMI: 0.7276, ASW: 0.5593, ARI: 0.6304, Delta: 0.0121
# Epoch 68 completed. Average Loss: 0.1216
# [Epoch 69] ACC: 0.7986, NMI: 0.7304, ASW: 0.5684, ARI: 0.6333, Delta: 0.0071
# Epoch 69 completed. Average Loss: 0.1239
# [Epoch 70] ACC: 0.7994, NMI: 0.7323, ASW: 0.5762, ARI: 0.6344, Delta: 0.0051
# Epoch 70 completed. Average Loss: 0.1261
# [Epoch 71] ACC: 0.7987, NMI: 0.7329, ASW: 0.5815, ARI: 0.6338, Delta: 0.0040
# Epoch 71 completed. Average Loss: 0.1280
# [Epoch 72] ACC: 0.7977, NMI: 0.7329, ASW: 0.5851, ARI: 0.6326, Delta: 0.0038
# Epoch 72 completed. Average Loss: 0.1296
# [Epoch 73] ACC: 0.7955, NMI: 0.7320, ASW: 0.5887, ARI: 0.6298, Delta: 0.0042
# Epoch 73 completed. Average Loss: 0.1313
# [Epoch 74] ACC: 0.7933, NMI: 0.7311, ASW: 0.5917, ARI: 0.6271, Delta: 0.0041
# Epoch 74 completed. Average Loss: 0.1328
# [Epoch 75] ACC: 0.7904, NMI: 0.7301, ASW: 0.5902, ARI: 0.6236, Delta: 0.0043
# Epoch 75 completed. Average Loss: 0.1341
# [Epoch 76] ACC: 0.7873, NMI: 0.7286, ASW: 0.5897, ARI: 0.6202, Delta: 0.0044
# Epoch 76 completed. Average Loss: 0.1353
# [Epoch 77] ACC: 0.7845, NMI: 0.7272, ASW: 0.5886, ARI: 0.6169, Delta: 0.0037
# Epoch 77 completed. Average Loss: 0.1368
# [Epoch 78] ACC: 0.7823, NMI: 0.7264, ASW: 0.5869, ARI: 0.6148, Delta: 0.0029
# Epoch 78 completed. Average Loss: 0.1386
# [Epoch 79] ACC: 0.7799, NMI: 0.7254, ASW: 0.5840, ARI: 0.6130, Delta: 0.0047
# Epoch 79 completed. Average Loss: 0.1404
# [Epoch 80] ACC: 0.7783, NMI: 0.7252, ASW: 0.5829, ARI: 0.6126, Delta: 0.0057
# Epoch 80 completed. Average Loss: 0.1420
# [Epoch 81] ACC: 0.7796, NMI: 0.7256, ASW: 0.5843, ARI: 0.6147, Delta: 0.0049
# Epoch 81 completed. Average Loss: 0.1435
# [Epoch 82] ACC: 0.7832, NMI: 0.7274, ASW: 0.5890, ARI: 0.6191, Delta: 0.0044
# Epoch 82 completed. Average Loss: 0.1444
# [Epoch 83] ACC: 0.7901, NMI: 0.7302, ASW: 0.5971, ARI: 0.6265, Delta: 0.0081
# Epoch 83 completed. Average Loss: 0.1446
# [Epoch 84] ACC: 0.7959, NMI: 0.7325, ASW: 0.6078, ARI: 0.6327, Delta: 0.0072
# Epoch 84 completed. Average Loss: 0.1440
# [Epoch 85] ACC: 0.8016, NMI: 0.7351, ASW: 0.6166, ARI: 0.6396, Delta: 0.0069
# Epoch 85 completed. Average Loss: 0.1429
# [Epoch 86] ACC: 0.8050, NMI: 0.7366, ASW: 0.6241, ARI: 0.6435, Delta: 0.0045
# Epoch 86 completed. Average Loss: 0.1421
# [Epoch 87] ACC: 0.8074, NMI: 0.7377, ASW: 0.6296, ARI: 0.6466, Delta: 0.0035
# Epoch 87 completed. Average Loss: 0.1415
# [Epoch 88] ACC: 0.8080, NMI: 0.7376, ASW: 0.6326, ARI: 0.6473, Delta: 0.0019
# Epoch 88 completed. Average Loss: 0.1409
# [Epoch 89] ACC: 0.8080, NMI: 0.7373, ASW: 0.6338, ARI: 0.6473, Delta: 0.0013
# Epoch 89 completed. Average Loss: 0.1402
# [Epoch 90] ACC: 0.8083, NMI: 0.7372, ASW: 0.6336, ARI: 0.6480, Delta: 0.0018
# Epoch 90 completed. Average Loss: 0.1392
# [Epoch 91] ACC: 0.8079, NMI: 0.7367, ASW: 0.6325, ARI: 0.6477, Delta: 0.0011
# Epoch 91 completed. Average Loss: 0.1382
# [Epoch 92] ACC: 0.8079, NMI: 0.7366, ASW: 0.6309, ARI: 0.6479, Delta: 0.0011
# Epoch 92 completed. Average Loss: 0.1374
# [Epoch 93] ACC: 0.8079, NMI: 0.7364, ASW: 0.6293, ARI: 0.6479, Delta: 0.0016
# Epoch 93 completed. Average Loss: 0.1367
# [Epoch 94] ACC: 0.8082, NMI: 0.7368, ASW: 0.6287, ARI: 0.6482, Delta: 0.0015
# Epoch 94 completed. Average Loss: 0.1361
# [Epoch 95] ACC: 0.8079, NMI: 0.7367, ASW: 0.6270, ARI: 0.6477, Delta: 0.0016
# Epoch 95 completed. Average Loss: 0.1357
# [Epoch 96] ACC: 0.8072, NMI: 0.7360, ASW: 0.6252, ARI: 0.6465, Delta: 0.0023
# Epoch 96 completed. Average Loss: 0.1351
# [Epoch 97] ACC: 0.8065, NMI: 0.7361, ASW: 0.6253, ARI: 0.6457, Delta: 0.0018
# Epoch 97 completed. Average Loss: 0.1345
# [Epoch 98] ACC: 0.8062, NMI: 0.7359, ASW: 0.6247, ARI: 0.6453, Delta: 0.0013
# Epoch 98 completed. Average Loss: 0.1339
# [Epoch 99] ACC: 0.8064, NMI: 0.7360, ASW: 0.6260, ARI: 0.6457, Delta: 0.0011
# Epoch 99 completed. Average Loss: 0.1332
# [Epoch 100] ACC: 0.8070, NMI: 0.7363, ASW: 0.6267, ARI: 0.6462, Delta: 0.0013
# Epoch 100 completed. Average Loss: 0.1325
# [Epoch 101] ACC: 0.8076, NMI: 0.7365, ASW: 0.6274, ARI: 0.6469, Delta: 0.0017
# Epoch 101 completed. Average Loss: 0.1318
# [Epoch 102] ACC: 0.8084, NMI: 0.7366, ASW: 0.6276, ARI: 0.6478, Delta: 0.0022
# Epoch 102 completed. Average Loss: 0.1310
# [Epoch 103] ACC: 0.8102, NMI: 0.7376, ASW: 0.6289, ARI: 0.6502, Delta: 0.0025
# Epoch 103 completed. Average Loss: 0.1302
# [Epoch 104] ACC: 0.8111, NMI: 0.7376, ASW: 0.6293, ARI: 0.6513, Delta: 0.0021
# Epoch 104 completed. Average Loss: 0.1292
# [Epoch 105] ACC: 0.8118, NMI: 0.7377, ASW: 0.6284, ARI: 0.6525, Delta: 0.0025
# Epoch 105 completed. Average Loss: 0.1280
# [Epoch 106] ACC: 0.8124, NMI: 0.7379, ASW: 0.6286, ARI: 0.6534, Delta: 0.0022
# Epoch 106 completed. Average Loss: 0.1268
# [Epoch 107] ACC: 0.8132, NMI: 0.7376, ASW: 0.6283, ARI: 0.6541, Delta: 0.0026
# Epoch 107 completed. Average Loss: 0.1256
# [Epoch 108] ACC: 0.8139, NMI: 0.7373, ASW: 0.6278, ARI: 0.6549, Delta: 0.0026
# Epoch 108 completed. Average Loss: 0.1244
# [Epoch 109] ACC: 0.8142, NMI: 0.7370, ASW: 0.6273, ARI: 0.6557, Delta: 0.0026
# Epoch 109 completed. Average Loss: 0.1233
# [Epoch 110] ACC: 0.8143, NMI: 0.7368, ASW: 0.6292, ARI: 0.6558, Delta: 0.0018
# Epoch 110 completed. Average Loss: 0.1222
# [Epoch 111] ACC: 0.8143, NMI: 0.7363, ASW: 0.6289, ARI: 0.6555, Delta: 0.0018
# Epoch 111 completed. Average Loss: 0.1212
# [Epoch 112] ACC: 0.8139, NMI: 0.7356, ASW: 0.6259, ARI: 0.6551, Delta: 0.0023
# Epoch 112 completed. Average Loss: 0.1203
# [Epoch 113] ACC: 0.8134, NMI: 0.7350, ASW: 0.6248, ARI: 0.6545, Delta: 0.0028
# Epoch 113 completed. Average Loss: 0.1196
# [Epoch 114] ACC: 0.8123, NMI: 0.7341, ASW: 0.6237, ARI: 0.6534, Delta: 0.0037
# Epoch 114 completed. Average Loss: 0.1189
# [Epoch 115] ACC: 0.8111, NMI: 0.7328, ASW: 0.6224, ARI: 0.6519, Delta: 0.0029
# Epoch 115 completed. Average Loss: 0.1181
# [Epoch 116] ACC: 0.8100, NMI: 0.7316, ASW: 0.6212, ARI: 0.6507, Delta: 0.0038
# Epoch 116 completed. Average Loss: 0.1174
# [Epoch 117] ACC: 0.8091, NMI: 0.7307, ASW: 0.6198, ARI: 0.6499, Delta: 0.0037
# Epoch 117 completed. Average Loss: 0.1166
# [Epoch 118] ACC: 0.8076, NMI: 0.7294, ASW: 0.6177, ARI: 0.6485, Delta: 0.0039
# Epoch 118 completed. Average Loss: 0.1159
# [Epoch 119] ACC: 0.8061, NMI: 0.7279, ASW: 0.6159, ARI: 0.6471, Delta: 0.0047
# Epoch 119 completed. Average Loss: 0.1152
# [Epoch 120] ACC: 0.8053, NMI: 0.7269, ASW: 0.6151, ARI: 0.6465, Delta: 0.0052
# Epoch 120 completed. Average Loss: 0.1144
# [Epoch 121] ACC: 0.8037, NMI: 0.7248, ASW: 0.6144, ARI: 0.6446, Delta: 0.0069
# Epoch 121 completed. Average Loss: 0.1139
# [Epoch 122] ACC: 0.8021, NMI: 0.7234, ASW: 0.6143, ARI: 0.6430, Delta: 0.0083
# Epoch 122 completed. Average Loss: 0.1132
# [Epoch 123] ACC: 0.7995, NMI: 0.7216, ASW: 0.6136, ARI: 0.6406, Delta: 0.0095
# Epoch 123 completed. Average Loss: 0.1125
# [Epoch 124] ACC: 0.7952, NMI: 0.7191, ASW: 0.6157, ARI: 0.6364, Delta: 0.0123
# Epoch 124 completed. Average Loss: 0.1121
# [Epoch 125] ACC: 0.7893, NMI: 0.7161, ASW: 0.6166, ARI: 0.6311, Delta: 0.0148
# Epoch 125 completed. Average Loss: 0.1118
# [Epoch 126] ACC: 0.7821, NMI: 0.7125, ASW: 0.6189, ARI: 0.6245, Delta: 0.0188
# Epoch 126 completed. Average Loss: 0.1123
# [Epoch 127] ACC: 0.7716, NMI: 0.7082, ASW: 0.6219, ARI: 0.6164, Delta: 0.0244
# Epoch 127 completed. Average Loss: 0.1131
# [Epoch 128] ACC: 0.7578, NMI: 0.7033, ASW: 0.6251, ARI: 0.6067, Delta: 0.0311
# Epoch 128 completed. Average Loss: 0.1146
# [Epoch 129] ACC: 0.7420, NMI: 0.6991, ASW: 0.6278, ARI: 0.5979, Delta: 0.0398
# Epoch 129 completed. Average Loss: 0.1163
# [Epoch 130] ACC: 0.7228, NMI: 0.6953, ASW: 0.6338, ARI: 0.5908, Delta: 0.0487
# Epoch 130 completed. Average Loss: 0.1178
# [Epoch 131] ACC: 0.7145, NMI: 0.6934, ASW: 0.6381, ARI: 0.5871, Delta: 0.0563
# Epoch 131 completed. Average Loss: 0.1184
# [Epoch 132] ACC: 0.7196, NMI: 0.6935, ASW: 0.6367, ARI: 0.5868, Delta: 0.0570
# Epoch 132 completed. Average Loss: 0.1180
# [Epoch 133] ACC: 0.7264, NMI: 0.6954, ASW: 0.6373, ARI: 0.5889, Delta: 0.0562
# Epoch 133 completed. Average Loss: 0.1175
# [Epoch 134] ACC: 0.7267, NMI: 0.6977, ASW: 0.6368, ARI: 0.5911, Delta: 0.0493
# Epoch 134 completed. Average Loss: 0.1168
# [Epoch 135] ACC: 0.7368, NMI: 0.7001, ASW: 0.6342, ARI: 0.5943, Delta: 0.0432
# Epoch 135 completed. Average Loss: 0.1161
# [Epoch 136] ACC: 0.7500, NMI: 0.7027, ASW: 0.6321, ARI: 0.5977, Delta: 0.0374
# Epoch 136 completed. Average Loss: 0.1152
# [Epoch 137] ACC: 0.7591, NMI: 0.7053, ASW: 0.6326, ARI: 0.6006, Delta: 0.0291
# Epoch 137 completed. Average Loss: 0.1144
# [Epoch 138] ACC: 0.7644, NMI: 0.7066, ASW: 0.6332, ARI: 0.6025, Delta: 0.0215
# Epoch 138 completed. Average Loss: 0.1139
# [Epoch 139] ACC: 0.7687, NMI: 0.7082, ASW: 0.6323, ARI: 0.6048, Delta: 0.0176
# Epoch 139 completed. Average Loss: 0.1134
# [Epoch 140] ACC: 0.7706, NMI: 0.7097, ASW: 0.6333, ARI: 0.6065, Delta: 0.0126
# Epoch 140 completed. Average Loss: 0.1131
# [Epoch 141] ACC: 0.7710, NMI: 0.7105, ASW: 0.6343, ARI: 0.6073, Delta: 0.0069
# Epoch 141 completed. Average Loss: 0.1130
# [Epoch 142] ACC: 0.7708, NMI: 0.7107, ASW: 0.6357, ARI: 0.6074, Delta: 0.0028
# Epoch 142 completed. Average Loss: 0.1131
# [Epoch 143] ACC: 0.7709, NMI: 0.7105, ASW: 0.6366, ARI: 0.6075, Delta: 0.0033
# Epoch 143 completed. Average Loss: 0.1135
# [Epoch 144] ACC: 0.7709, NMI: 0.7099, ASW: 0.6371, ARI: 0.6074, Delta: 0.0089
# Epoch 144 completed. Average Loss: 0.1143
# [Epoch 145] ACC: 0.7683, NMI: 0.7087, ASW: 0.6407, ARI: 0.6061, Delta: 0.0140
# Epoch 145 completed. Average Loss: 0.1156
# [Epoch 146] ACC: 0.7632, NMI: 0.7071, ASW: 0.6408, ARI: 0.6037, Delta: 0.0207
# Epoch 146 completed. Average Loss: 0.1173
# [Epoch 147] ACC: 0.7541, NMI: 0.7045, ASW: 0.6425, ARI: 0.6003, Delta: 0.0288
# Epoch 147 completed. Average Loss: 0.1200
# [Epoch 148] ACC: 0.7417, NMI: 0.7013, ASW: 0.6460, ARI: 0.5961, Delta: 0.0405
# Epoch 148 completed. Average Loss: 0.1237
# [Epoch 149] ACC: 0.7236, NMI: 0.6975, ASW: 0.6502, ARI: 0.5912, Delta: 0.0545
# Epoch 149 completed. Average Loss: 0.1278
# Final ACC: 0.7236